[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/finique/praxis-reproductions/blob/main/361154-network-momentum.ipynb)

# Network Momentum across Asset Classes — reproduction

**Xingyue (Stacy) Pu, Stephen Roberts, Xiaowen Dong & Stefan Zohren** · Oxford-Man Institute · [arXiv 2308.11294](https://arxiv.org/abs/2308.11294) (2023)

The paper learns a per-day *graph* linking futures whose momentum co-moves, then trades each asset on its **neighbours'** momentum ("network momentum", GMOM). Headline: Sharpe **1.51** / return **22.2%** (2000–2022, 64 Pinnacle futures) — and, the crux, GMOM beats trading each asset's *own* momentum (individual-momentum LinReg, 1.13).

This notebook runs the **same data and logic** as our engine and lands on the report's exact numbers. It loads the roll-adjusted continuous-futures reference panel (exchange daily settlements from CME Globex / ICE, via Databento, ratio back-adjusted at every roll — 40 of the 64 contracts, daily 2010-06→2026-06; no free pre-2010 daily futures exists, so the out-of-sample is 2020–2026, with 2020–2022 overlapping the paper and 2023–2026 post-publication). It then rebuilds the 8 momentum features, learns the daily ensemble graphs (Kalofolias log-degree convex model — the paper's Eq 4, same CVXPY they use), propagates network momentum, fits the cross-sectional regression, and prints **claimed → measured**.

**What you'll see:** GMOM measures **0.62**, not 1.51; the network edge over individual momentum is real in direction (+0.58) but statistically insignificant; and a **plain 12-month time-series momentum the paper never shows scores 0.80 — it beats the graph-learning model** on identical data. Verdict: **Overstated**. The full C0–C7 gate suite (deflation, cost, crisis, spanning) lives in the [report](https://market-graphs.com/praxis/papers/361154).


In [ ]:
# Pinned to the engine's versions so the numbers reproduce exactly.
%pip install -q numpy==2.1.2 pandas==2.2.3 scipy==1.14.1 cvxpy==1.9.2 clarabel pyarrow

In [ ]:
"""Praxis reproduction — paper 361154 (arXiv 2308.11294), Network Momentum
across Asset Classes (Pu, Roberts, Dong & Zohren, Oxford-Man Institute).

Self-contained port of the engine blocks (momentum_features, graph_learn,
network_momentum) + the exemplar strategy. Loads the frozen roll-adjusted
reference panel (40 futures, daily 2010-06→2026-06) and prints claimed→measured.
Deterministic: identical input → identical output (OLS + convex solve, no RNG).

Reproduces the report's OOS 2020-2026 numbers: GMOM (network momentum) Sharpe
0.62 vs a plain 12-month TSMOM 0.80 — the naive trend the paper never shows,
which beats the graph-learning model on identical data.
"""
import json
import urllib.request

import numpy as np
import pandas as pd
import cvxpy as cp
import scipy.sparse as sp
import warnings; warnings.filterwarnings('ignore')

REPO = "https://raw.githubusercontent.com/finique/praxis-reproductions/main"
WINDOWS = (252, 504, 756, 1008, 1260)
ALPHA, BETA = 0.1, 0.1
TARGET_VOL, PPY = 0.15, 252
SPLITS = [("2019-12-31", "2022-12-31"), ("2022-12-31", "2026-06-30")]
OOS_START = "2020-01-01"


# ------------------------------------------------------------- data ------------
def load_panel():
    # The reference panel derives from licensed Databento (CME GLBX + ICE) market
    # data and is NOT redistributed with this repository. To run: rebuild the
    # panel against your own Databento subscription — the same adapter pattern
    # Praxis ships to clients (your entitlement, your data) — or contact
    # MarketGraphs, then place the file next to this notebook.
    try:
        px = pd.read_parquet("361154-futures-panel.parquet")
    except FileNotFoundError:
        raise SystemExit(
            "361154-futures-panel.parquet not found — the panel is built from licensed "
            "Databento data and is not redistributed. Rebuild it under your own Databento "
            "entitlement, or contact MarketGraphs (see the note above).")
    # forward-fill holiday gaps within each contract's live span (mixed calendars)
    ff = px.ffill()
    return ff.where(ff.notna() & px.bfill().notna())


# ------------------------------------------------- 8 Baz momentum features -----
def _ewmstd(x, span):
    return x.ewm(span=span).std(bias=True)


def _winsor(f):
    mu = f.ewm(halflife=252).mean()
    sd = f.ewm(halflife=252).std(bias=True)
    return f.clip(lower=mu - 5 * sd, upper=mu + 5 * sd)


def build_features(prices):
    rets = prices.pct_change()
    sigma = rets.apply(lambda c: _ewmstd(c, 60))
    sig = sigma.where(sigma > 0)
    feats = {}
    for d in (1, 21, 63, 126, 252):
        feats[f"ret_{d}"] = _winsor((prices / prices.shift(d) - 1) / (sig * np.sqrt(d)))
    for s, l in ((8, 24), (16, 48), (32, 96)):
        macd = prices.ewm(alpha=1 / s, adjust=False).mean() - prices.ewm(alpha=1 / l, adjust=False).mean()
        norm = macd / prices.rolling(63).std()
        feats[f"macd_{s}_{l}"] = _winsor(norm / norm.rolling(252).std())
    names = list(feats)
    fwd = rets.shift(-1) / sig
    return feats, names, fwd, sigma * np.sqrt(PPY), rets.shift(-1)


# ----------------------------------------- Kalofolias log-degree graph ---------
_PROB = {}


def learn_graph(Z, alpha, beta):
    n = Z.shape[0]
    if n < 2:
        return np.zeros((n, n))
    key = n
    if key not in _PROB:
        ii, jj = np.triu_indices(n, 1)
        m = ii.size
        rows = np.concatenate([ii, jj]); cols = np.concatenate([np.arange(m), np.arange(m)])
        S = sp.csr_matrix((np.ones(2 * m), (rows, cols)), shape=(n, m))
        z = cp.Parameter(m, nonneg=True); w = cp.Variable(m, nonneg=True)
        prob = cp.Problem(cp.Minimize(z @ w - alpha * cp.sum(cp.log(S @ w)) + beta * cp.sum_squares(w)))
        _PROB[key] = (prob, z, w, ii, jj)
    prob, z, w, ii, jj = _PROB[key]
    z.value = Z[ii, jj].astype(float)
    prob.solve(solver=cp.CLARABEL, warm_start=True)
    wv = np.maximum(np.asarray(w.value, float), 0.0)
    A = np.zeros((n, n)); A[ii, jj] = wv; A[jj, ii] = wv
    return A


def normalize_adj(A):
    d = A.sum(1); inv = np.zeros_like(d); nz = d > 0; inv[nz] = 1 / np.sqrt(d[nz])
    return A * inv[:, None] * inv[None, :]


def pairwise_sqdist(V):
    g = V @ V.T; dg = np.diag(g)
    Z = dg[:, None] + dg[None, :] - 2 * g
    np.fill_diagonal(Z, 0.0)
    return np.maximum(Z, 0.0)


# ------------------------------------------------ graphs + network features ----
def build_graphs(U, stride=1):
    T, N, F = U.shape
    finite = np.isfinite(U).all(2)
    mx = max(WINDOWS)
    graphs, first = {}, None
    for it in range(mx - 1, T):
        node = finite[it - mx + 1:it + 1].all(0)
        if node.sum() < 3:
            continue
        if first is None:
            first = it
        if (it - first) % stride:
            continue
        idx = np.where(node)[0]
        mats = []
        for w in WINDOWS:
            V = U[it - w + 1:it + 1][:, idx, :].transpose(1, 0, 2).reshape(len(idx), w * F)
            mats.append(learn_graph(pairwise_sqdist(V), ALPHA, BETA))
        graphs[it] = (idx, normalize_adj(np.mean(mats, 0)))
    return graphs


def network_features(U, graphs):
    T, N, F = U.shape
    out = np.full((T, N, F), np.nan)
    gk = sorted(graphs); gi = 0
    for it in range(gk[0], T):
        while gi + 1 < len(gk) and gk[gi + 1] <= it:
            gi += 1
        idx, A = graphs[gk[gi]]
        out[it][idx, :] = A @ np.nan_to_num(U[it][idx, :])
    return out


# ------------------------------------------------------ OLS → positions --------
def fit_predict(feat, target, dates):
    T, N, F = feat.shape
    pos = np.full((T, N), np.nan)
    for tr_end, te_end in SPLITS:
        te0 = dates.searchsorted(pd.Timestamp(tr_end), "right")
        te1 = dates.searchsorted(pd.Timestamp(te_end), "right")
        X, y = [], []
        for it in range(0, te0):
            m = np.isfinite(feat[it]).all(1) & np.isfinite(target[it])
            if m.any():
                X.append(feat[it][m]); y.append(target[it][m])
        X = np.vstack(X); y = np.concatenate(y)
        mu, sd = X.mean(0), X.std(0); sd = np.where(sd > 0, sd, 1)
        A = np.column_stack([(X - mu) / sd, np.ones(len(X))])
        c, *_ = np.linalg.lstsq(A, y, rcond=None)
        for it in range(te0, te1):
            m = np.isfinite(feat[it]).all(1)
            if m.any():
                pos[it, np.where(m)[0]] = np.sign(((feat[it][m] - mu) / sd) @ c[:-1] + c[-1])
    return pos


def portfolio(pos, ret_fwd, sig_ann, dates):
    T, N = pos.shape
    out = np.full(T, np.nan)
    for it in range(T):
        m = np.isfinite(pos[it]) & np.isfinite(ret_fwd[it]) & np.isfinite(sig_ann[it]) & (sig_ann[it] > 0)
        if m.sum():
            out[it] = (pos[it][m] * TARGET_VOL / sig_ann[it][m] * ret_fwd[it][m]).mean()
    return pd.Series(out, dates).shift(1).dropna()


def vol_target(port):
    vol = port.ewm(span=60).std(bias=True) * np.sqrt(PPY)
    return ((TARGET_VOL / vol.shift(1)).clip(upper=10) * port).dropna()


def sharpe(r, a=OOS_START, b=None):
    x = r.loc[a:b].dropna()
    return float(x.mean() / x.std(ddof=1) * np.sqrt(PPY))


# ------------------------------------------------------------------ main -------
def main():
    print("Loading the roll-adjusted reference panel (40 futures, 2010-2026)…")
    prices = load_panel()
    assets = list(prices.columns)
    dates = prices.index
    feats, names, fwd, sig_ann, ret_fwd = build_features(prices)
    U = np.full((len(dates), len(assets), 8), np.nan)
    for k, nm in enumerate(names):
        U[:, :, k] = feats[nm].values
    tgt = fwd.values; sa = sig_ann.values; rf = ret_fwd.values

    print("Learning the daily ensemble graphs (Kalofolias log-degree, ~2-3 min)…")
    graphs = build_graphs(U, stride=1)
    Ut = network_features(U, graphs)

    gmom = vol_target(portfolio(fit_predict(Ut, tgt, dates), rf, sa, dates))
    linreg = vol_target(portfolio(fit_predict(U, tgt, dates), rf, sa, dates))
    r252 = feats["ret_252"].values
    tsmom_pos = np.sign(r252); tsmom_pos[:min(graphs)] = np.nan
    tsmom = vol_target(portfolio(tsmom_pos, rf, sa, dates))

    print("\n=== 361154 — Network Momentum across Asset Classes ===")
    print(f"{'strategy':22}{'measured SR':>12}{'claimed':>10}")
    print(f"{'GMOM (network mom.)':22}{sharpe(gmom):12.3f}{1.511:>10}")
    print(f"{'LinReg (individual)':22}{sharpe(linreg):12.3f}{1.128:>10}")
    print(f"{'Plain 12m TSMOM *':22}{sharpe(tsmom):12.3f}{'—':>10}")
    print("  * the naive trend rule the paper never shows — it BEATS the graph model")
    up = sharpe(gmom) - sharpe(linreg)
    print(f"\nnetwork uplift GMOM−LinReg = {up:+.3f} (claimed +0.383) — but not significant")
    print(f"a plain 12-month trend rule ({sharpe(tsmom):.2f}) beats the graph model "
          f"({sharpe(gmom):.2f}) on identical data.")
    print("VERDICT: OVERSTATED. Full C0–C7 gate suite → market-graphs.com/praxis/papers/361154")
    return gmom, tsmom

In [ ]:
gmom, tsmom = main()

In [ ]:
import matplotlib.pyplot as plt
eqg = (1 + gmom.loc['2020-01-01':]).cumprod()
eqt = (1 + tsmom.reindex(gmom.index).fillna(0).loc['2020-01-01':]).cumprod()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eqg.index, eqg.values, label='GMOM network momentum (measured 0.62)', lw=1.4)
ax.plot(eqt.index, eqt.values, label='Plain 12-month TSMOM (measured 0.80)', lw=1.4)
ax.set_title('Network momentum vs the naive trend rule it never shows — 2020-2026, 15% vol')
ax.set_ylabel('Growth of 1'); ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()